# Skin Condition CNN Training

This notebook trains a transfer-learning CNN for the MediTech skin-condition detector.

It uses the same dataset and model artifact paths as the Flask app, so once training finishes you can restart `app.py` and the UI will use the CNN automatically.

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\OM\Desktop\Emergency-Response-Hub-2\Emergency-Response-Hub-2")
BACKEND_DIR = PROJECT_ROOT / "artifacts" / "ai-backend"

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("BACKEND_DIR =", BACKEND_DIR)
print("exists =", BACKEND_DIR.exists())


PROJECT_ROOT = C:\Users\OM\Desktop\Emergency-Response-Hub-2\Emergency-Response-Hub-2
BACKEND_DIR = C:\Users\OM\Desktop\Emergency-Response-Hub-2\Emergency-Response-Hub-2\artifacts\ai-backend
exists = True


In [4]:
from ml.dataset_metadata import SKIN_DATASET_DIR, SKIN_CNN_MODEL_PATH

DATASET_DIR = Path(SKIN_DATASET_DIR)
MODEL_PATH = Path(SKIN_CNN_MODEL_PATH)
REPORT_PATH = MODEL_PATH.with_name("skin_condition_cnn_report.json")
CONFUSION_MATRIX_PATH = MODEL_PATH.with_name("skin_condition_cnn_confusion_matrix.csv")

DATASET_DIR, MODEL_PATH


(WindowsPath('C:/Users/OM/Desktop/Emergency-Response-Hub-2/Emergency-Response-Hub-2/artifacts/ai-backend/artifacts/datasets/skin_conditions'),
 WindowsPath('C:/Users/OM/Desktop/Emergency-Response-Hub-2/Emergency-Response-Hub-2/artifacts/ai-backend/artifacts/models/skin_condition_cnn.pth'))

In [5]:
from collections import Counter, defaultdict
import random

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
RESERVED_SPLITS = {"train", "test", "testing", "val", "valid"}

def normalize_split_name(name: str) -> str:
    if name == "testing":
        return "test"
    if name == "valid":
        return "val"
    return name

def discover_items(dataset_dir: Path):
    grouped = {"train": [], "test": [], "val": []}
    for split_name in ("train", "test", "testing", "val", "valid"):
        split_dir = dataset_dir / split_name
        if not split_dir.exists():
            continue
        normalized = normalize_split_name(split_name)
        for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
            for image_path in sorted(class_dir.rglob("*")):
                if image_path.is_file() and image_path.suffix.lower() in VALID_EXTENSIONS:
                    grouped[normalized].append((image_path, class_dir.name))

    flat_by_class = defaultdict(list)
    for class_dir in sorted(path for path in dataset_dir.iterdir() if path.is_dir() and path.name not in RESERVED_SPLITS):
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.is_file() and image_path.suffix.lower() in VALID_EXTENSIONS:
                flat_by_class[class_dir.name].append(image_path)

    rng = random.Random(42)
    for label, paths in flat_by_class.items():
        rng.shuffle(paths)
        if len(paths) >= 10:
            val_count = max(1, int(len(paths) * 0.15))
            grouped["test"].extend((path, label) for path in paths[:val_count])
            grouped["train"].extend((path, label) for path in paths[val_count:])
        else:
            grouped["train"].extend((path, label) for path in paths)
    return grouped

grouped = discover_items(DATASET_DIR)
for split_name, items in grouped.items():
    print(split_name, len(items))

train 23964
test 4254
val 0


In [6]:
class_counts = Counter(label for split in grouped.values() for _, label in split)
class_counts

Counter({'melanocytic_nevi': 7970,
         'basal_cell_carcinoma': 3536,
         'melanoma': 3140,
         'viral_warts_molluscum': 2103,
         'benign_keratosis_like_lesions': 2079,
         'psoriasis_lichen_planus': 2055,
         'eczema': 1890,
         'seborrheic_keratoses': 1847,
         'fungal_infection': 1702,
         'atopic_dermatitis': 1257,
         'acne': 213,
         'actinic_keratosis': 213,
         'rosacea': 213})

In [7]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image

device = torch.device("cpu")
print(torch.__version__, device)

2.9.1+cpu cpu


In [8]:
class SkinImageDataset(Dataset):
    def __init__(self, items, class_to_idx, transform):
        self.items = items
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]
        image = Image.open(path).convert("RGB")
        return self.transform(image), self.class_to_idx[label]

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

In [9]:
train_items = grouped["train"]
test_items = grouped["test"] or grouped["val"]
class_names = sorted({label for _, label in train_items + test_items})
class_to_idx = {label: idx for idx, label in enumerate(class_names)}

train_ds = SkinImageDataset(train_items, class_to_idx, train_tf)
test_ds = SkinImageDataset(test_items, class_to_idx, eval_tf)

train_counts = Counter(label for _, label in train_items)
sample_weights = [1.0 / train_counts[label] for _, label in train_items]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

len(class_names), len(train_ds), len(test_ds)

(13, 23964, 4254)

In [10]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(model.fc.in_features, len(class_names)),
)
model = model.to(device)

class_weights = torch.tensor([1.0 / train_counts.get(label, 1) for label in class_names], dtype=torch.float32, device=device)
class_weights = class_weights / class_weights.sum() * len(class_names)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW([param for param in model.parameters() if param.requires_grad], lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
from sklearn.metrics import classification_report, confusion_matrix

def train_epoch(model, loader):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0
    for batch_idx, (images, targets) in enumerate(loader, start=1):
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        total += targets.size(0)
        correct += (preds == targets).sum().item()
        if batch_idx % 50 == 0:
            print(f"batch {batch_idx}/{len(loader)} loss={running_loss / batch_idx:.4f} acc={correct / max(1,total):.4f}")
    return running_loss / max(1, len(loader)), correct / max(1, total)

def evaluate(model, loader):
    model.eval()
    losses = []
    preds_all = []
    targets_all = []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            logits = model(images)
            loss = criterion(logits, targets)
            losses.append(loss.item())
            preds = logits.argmax(dim=1)
            preds_all.extend(preds.cpu().tolist())
            targets_all.extend(targets.cpu().tolist())
    report = classification_report(targets_all, preds_all, labels=list(range(len(class_names))), target_names=class_names, output_dict=True, zero_division=0)
    return {
        "loss": sum(losses) / max(1, len(losses)),
        "accuracy": report.get("accuracy", 0.0),
        "macro_f1": report.get("macro avg", {}).get("f1-score", 0.0),
        "report": report,
        "predictions": preds_all,
        "targets": targets_all,
    }

In [ ]:
EPOCHS = 6
best_state = None
best_eval = None

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader)
    eval_result = evaluate(model, test_loader)
    scheduler.step()
    print(f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={eval_result['loss']:.4f} val_acc={eval_result['accuracy']:.4f} val_macro_f1={eval_result['macro_f1']:.4f}")
    if best_eval is None or (eval_result['macro_f1'], eval_result['accuracy']) > (best_eval['macro_f1'], best_eval['accuracy']):
        best_eval = eval_result
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

best_eval

Epoch 1/6
batch 50/1498 loss=1.6290 acc=0.2550
batch 100/1498 loss=1.4114 acc=0.3081
batch 150/1498 loss=1.2563 acc=0.3412
batch 200/1498 loss=1.2105 acc=0.3528
batch 250/1498 loss=1.1653 acc=0.3718
batch 300/1498 loss=1.1611 acc=0.3762
batch 350/1498 loss=1.1165 acc=0.3912
batch 400/1498 loss=1.0943 acc=0.4005
batch 450/1498 loss=1.0795 acc=0.4056
batch 500/1498 loss=1.0603 acc=0.4088
batch 550/1498 loss=1.0275 acc=0.4172
batch 600/1498 loss=1.0118 acc=0.4222
batch 650/1498 loss=0.9869 acc=0.4294
batch 700/1498 loss=0.9727 acc=0.4336
batch 750/1498 loss=0.9616 acc=0.4381
batch 800/1498 loss=0.9441 acc=0.4409
batch 850/1498 loss=0.9280 acc=0.4465
batch 900/1498 loss=0.9084 acc=0.4525
batch 950/1498 loss=0.8928 acc=0.4567
batch 1000/1498 loss=0.8790 acc=0.4611
batch 1050/1498 loss=0.8647 acc=0.4650
batch 1100/1498 loss=0.8538 acc=0.4678
batch 1150/1498 loss=0.8487 acc=0.4709
batch 1200/1498 loss=0.8389 acc=0.4739
batch 1250/1498 loss=0.8271 acc=0.4772
batch 1300/1498 loss=0.8195 acc=0.4

In [ ]:
checkpoint = {
    "architecture": "resnet18",
    "classes": class_names,
    "state_dict": best_state,
}
torch.save(checkpoint, MODEL_PATH)
print(MODEL_PATH)

In [ ]:
matrix = confusion_matrix(best_eval['targets'], best_eval['predictions'], labels=list(range(len(class_names))))
matrix[:5, :5]

In [ ]:
import pandas as pd

df = pd.DataFrame(matrix, index=class_names, columns=class_names)
df.to_csv(CONFUSION_MATRIX_PATH)
pd.DataFrame(best_eval['report']).T.to_json(REPORT_PATH, indent=2)
df

After saving the checkpoint, restart the Flask app:

```powershell
python artifacts/ai-backend/app.py
```

The app will use `skin_condition_cnn.pth` automatically.